# SpaceX Falcon 9 — SQL Exploratory Data Analysis

## Project purpose
Use SQL and Pandas to answer operational questions about Falcon 9 launches, payloads, launch sites, mission outcomes, and first-stage landing outcomes.

**Skills demonstrated:** data loading, data validation, SQLite, SQL aggregation, filtering, subqueries, date handling, and translating query results into business observations.

> This notebook uses the IBM Skills Network SpaceX launch dataset. Run the notebook from top to bottom. The analysis is descriptive; it does not establish causation.

In [ ]:
# Load the dataset and create an in-memory SQLite database.
import sqlite3
import pandas as pd

DATA_URL = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_2/data/Spacex.csv"

df = pd.read_csv(DATA_URL)
df.columns = df.columns.str.strip()

# Standardize the table's column names only where needed by SQL.
df.to_sql("spacex", sqlite3.connect(":memory:"), if_exists="replace", index=False)
conn = sqlite3.connect(":memory:")
df.to_sql("spacex", conn, if_exists="replace", index=False)

print(f"Rows: {len(df):,} | Columns: {len(df.columns)}")
display(df.head())
print("\nColumn names:")
print(df.columns.tolist())

## Data quality checks

Check for missing dates and inspect the key categorical fields before interpreting query results.

In [ ]:
display(df.isna().sum().sort_values(ascending=False).to_frame("missing_values"))
display(df[["LAUNCH_SITE", "MISSION_OUTCOME", "Landing_Outcome"]].nunique().to_frame("unique_values"))

## SQL analysis

The queries below use the SQLite table created above. Each result is displayed as a DataFrame for easier review.

In [ ]:
# 1. Which launch sites appear in the dataset?
q1 = """
SELECT DISTINCT LAUNCH_SITE
FROM spacex
WHERE LAUNCH_SITE IS NOT NULL
ORDER BY LAUNCH_SITE;
"""
display(pd.read_sql_query(q1, conn))

In [ ]:
# 2. Show five launches from sites whose names begin with CCA.
q2 = """
SELECT *
FROM spacex
WHERE LAUNCH_SITE LIKE 'CCA%'
LIMIT 5;
"""
display(pd.read_sql_query(q2, conn))

In [ ]:
# 3. What is the total payload mass launched for NASA (CRS)?
q3 = """
SELECT SUM(PAYLOAD_MASS__KG_) AS total_payload_mass_kg
FROM spacex
WHERE CUSTOMER = 'NASA (CRS)';
"""
display(pd.read_sql_query(q3, conn))

In [ ]:
# 4. What is the average payload mass for booster version F9 v1.1?
q4 = """
SELECT AVG(PAYLOAD_MASS__KG_) AS average_payload_mass_kg
FROM spacex
WHERE BOOSTER_VERSION = 'F9 v1.1';
"""
display(pd.read_sql_query(q4, conn))

In [ ]:
# 5. What is the earliest successful ground-pad landing?
q5 = """
SELECT MIN(DATE) AS earliest_successful_ground_pad_landing
FROM spacex
WHERE Landing_Outcome = 'Success (ground pad)';
"""
display(pd.read_sql_query(q5, conn))

In [ ]:
# 6. Which booster versions had successful drone-ship landings with payloads from 4,000 to 6,000 kg?
q6 = """
SELECT DISTINCT BOOSTER_VERSION, PAYLOAD_MASS__KG_
FROM spacex
WHERE Landing_Outcome = 'Success (drone ship)'
  AND PAYLOAD_MASS__KG_ BETWEEN 4000 AND 6000
ORDER BY PAYLOAD_MASS__KG_;
"""
display(pd.read_sql_query(q6, conn))

In [ ]:
# 7. How many launches fall into each mission outcome?
q7 = """
SELECT MISSION_OUTCOME, COUNT(*) AS launch_count
FROM spacex
GROUP BY MISSION_OUTCOME
ORDER BY launch_count DESC;
"""
display(pd.read_sql_query(q7, conn))

In [ ]:
# 8. Which booster version(s) carried the maximum payload mass?
q8 = """
SELECT BOOSTER_VERSION, PAYLOAD_MASS__KG_
FROM spacex
WHERE PAYLOAD_MASS__KG_ = (
    SELECT MAX(PAYLOAD_MASS__KG_) FROM spacex
);
"""
display(pd.read_sql_query(q8, conn))

In [ ]:
# 9. Which booster versions had drone-ship landing failures during 2015?
q9 = """
SELECT DATE, BOOSTER_VERSION, LAUNCH_SITE, Landing_Outcome
FROM spacex
WHERE DATE >= '2015-01-01'
  AND DATE < '2016-01-01'
  AND Landing_Outcome = 'Failure (drone ship)'
ORDER BY DATE;
"""
display(pd.read_sql_query(q9, conn))

In [ ]:
# 10. Rank landing outcomes between 2010-06-04 and 2017-03-20.
q10 = """
SELECT Landing_Outcome, COUNT(*) AS outcome_count
FROM spacex
WHERE DATE BETWEEN '2010-06-04' AND '2017-03-20'
GROUP BY Landing_Outcome
ORDER BY outcome_count DESC;
"""
display(pd.read_sql_query(q10, conn))

## Key takeaways

Use the query outputs above to write 3–5 concise observations. For each observation, cite the specific table or metric that supports it. Avoid claiming that a launch site, payload size, or booster version *caused* an outcome based on descriptive SQL alone.

### Suggested business questions
- Which launch site has the highest number of records?
- Which landing outcome occurs most frequently?
- How does payload mass vary across customers or booster versions?
- What operational patterns would be worth exploring with a predictive model?

## Reproducibility
- Source: IBM Skills Network SpaceX Falcon 9 launch dataset (URL in the first code cell).
- Database: SQLite in-memory database; no local database file or legacy SQLAlchemy installation required.
- Run all cells from top to bottom to regenerate results.